In [11]:
import os
import base64
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from typing import TypedDict
from langchain_ollama import ChatOllama

# GMAIL SETUP 
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def get_gmail_service():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

#LLM 
llm = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

print("Setup done.")

Setup done.


LangGraph email agent

In [ ]:
# STATE
class EmailState(TypedDict):
    recipient_email: str
    recipient_name: str
    context: str
    subject: str
    body: str
    iterations: int
    approved: bool
    sent: bool

# NODE 1: DRAFT
def draft_email(state: EmailState) -> EmailState:
    print(f"\n--- DRAFT NODE (iteration {state['iterations'] + 1}) ---")
    
    response = llm.invoke([
        HumanMessage(content=f"""Write a professional cold email to {state['recipient_name']}.

Context: {state['context']}

Write the subject line on the first line starting with "Subject: "
Then write the email body after a blank line.
Keep it between 150-200 words. End with a clear call to action.
Sign off with "Best regards, Mudil Goel" at the end.""")
    ])
    
    content = response.content.strip()
    lines = content.split('\n')
    
    subject = ""
    body_lines = []
    found_subject = False
    
    for i, line in enumerate(lines):
        if line.lower().startswith("subject:"):
            subject = line.split(":", 1)[1].strip()
            found_subject = True
        elif found_subject and line.strip():
            body_lines.append(line)
    
    body = "\n".join(body_lines).strip()
    
    print(f"Subject: {subject}")
    print(f"Body word count: {len(body.split())}")
    print(f"Body preview: {body[:100]}...")
    
    return {
        **state,
        "subject": subject,
        "body": body,
        "iterations": state["iterations"] + 1
    }

#NODE 2: REVIEW 
def review_email(state: EmailState) -> EmailState:
    print(f"\n--- REVIEW NODE ---")
    
    word_count = len(state["body"].split())
    has_subject = len(state["subject"]) > 5
    has_enough_words = word_count >= 100
    
    approved = has_subject and has_enough_words
    
    print(f"Subject present: {has_subject}")
    print(f"Word count: {word_count} ({'OK' if has_enough_words else 'TOO SHORT'})")
    print(f"Review: {'APPROVED' if approved else 'REJECTED — redrafting'}")
    
    return {
        **state,
        "approved": approved,
        "iterations": state["iterations"] + 1
    }

# NODE 3: SEND
def send_email(state: EmailState) -> EmailState:
    print(f"\n--- SEND NODE ---")
    
    service = get_gmail_service()
    
    message = MIMEText(state["body"])
    message['to'] = state["recipient_email"]
    message['subject'] = state["subject"]
    
    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    
    service.users().messages().send(
        userId='me',
        body={'raw': raw}
    ).execute()
    
    print(f"Email sent to {state['recipient_email']} ✅")
    
    return {
        **state,
        "sent": True
    }

# CONDITIONAL EDGE
def condition(state: EmailState) -> str:
    if state["approved"]:
        return "send"
    elif state["iterations"] >= 5:
        print("Max iterations reached, sending anyway.")
        return "send"
    else:
        return "draft"

#BUILD GRAPH
graph = StateGraph(EmailState)

graph.add_node("draft", draft_email)
graph.add_node("review", review_email)
graph.add_node("send", send_email)

graph.set_entry_point("draft")
graph.add_edge("draft", "review")
graph.add_conditional_edges(
    "review",
    condition,
    {
        "send": "send",
        "draft": "draft"
    }
)
graph.add_edge("send", END)

app = graph.compile()
print("Email agent graph compiled.")

Email agent graph compiled.


Running the Graph


In [13]:
# ===== RUN THE AGENT =====
initial_state = {
    "recipient_email": "mudilgoel@gmail.com",  # sending to yourself
    "recipient_name": "Prof Lee",
    "context": "I am a sophomore at IIT Bombay studying Electrical Engineering with a minor in AI & Data Science. I am interested in ML research internships and have built RAG pipelines, LangGraph agents, and worked on quantitative finance projects. I would love to discuss potential research opportunities.",
    "subject": "",
    "body": "",
    "iterations": 0,
    "approved": False,
    "sent": False
}

print("Starting Cold Email Agent...")
print(f"Recipient: {initial_state['recipient_name']} ({initial_state['recipient_email']})\n")

final_state = app.invoke(initial_state)

print("\n" + "="*60)
print("FINAL STATE:")
print("="*60)
print(f"Subject: {final_state['subject']}")
print(f"Approved: {final_state['approved']}")
print(f"Iterations: {final_state['iterations']}")
print(f"Sent: {final_state['sent']}")
print(f"\nEmail Body:\n{final_state['body']}")

Starting Cold Email Agent...
Recipient: Prof Lee (mudilgoel@gmail.com)


--- DRAFT NODE (iteration 1) ---
Subject: Exploring Research Opportunities in Machine Learning and AI
Body word count: 165
Body preview: Dear Prof Lee,
I am writing to express my interest in potential research opportunities under your su...

--- REVIEW NODE ---
Subject present: True
Word count: 165 (OK)
Review: APPROVED

--- SEND NODE ---
Email sent to mudilgoel@gmail.com ✅

FINAL STATE:
Subject: Exploring Research Opportunities in Machine Learning and AI
Approved: True
Iterations: 2
Sent: True

Email Body:
Dear Prof Lee,
I am writing to express my interest in potential research opportunities under your supervision at IIT Bombay. As a sophomore studying Electrical Engineering with a minor in AI & Data Science, I have been actively working on various projects that align with the cutting-edge technologies in machine learning.
My academic background has equipped me with a strong foundation in RAG pipelines and LangGr